# Fig S5F — TF motif accessibility per perturbation (pooled vs NTC)
Faithful reproduction of **08_PlotTFActivity Part D** (heatmap). Each perturbation vs pooled NTC, no state stratification, on the **pseudotime peak set** (114k peaks) → getMarkerFeatures Log2FC → OLS on the grouped motifs. Shown: the nonzero FDR-significant coefficient sub-matrix (TFs × perturbations), rows/cols hierarchically clustered, each cell labelled with its coefficient.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
# --- 08_PlotTFActivity Part D (cell 15), replicated (pseudotime-peak perturbation result) ---
pcf = load_matrix('PertTFActivity_ptpeaks_coefFDRsig.csv')
pcf = pcf.loc[(pcf != 0).any(axis=1), (pcf != 0).any(axis=0)]
print('nonzero sub-matrix:', pcf.shape, '(TF x perturbation)')
def order(df, axis):                                    # pheatmap default: complete linkage
    X = df.values if axis == 0 else df.values.T
    if X.shape[0] < 3: return np.arange(X.shape[0])
    return leaves_list(linkage(X, method='complete'))
pcf = pcf.iloc[order(pcf, 0), order(pcf, 1)]
lim = np.abs(pcf.values).max()
fig, ax = plt.subplots(figsize=(max(3.2, 0.5*pcf.shape[1]+1.5), max(2.6, 0.3*pcf.shape[0]+1)))
im = ax.imshow(pcf.values, aspect='auto', cmap=ACTIVITY_CMAP, vmin=-lim, vmax=lim)
ax.set_xticks(range(pcf.shape[1])); ax.set_xticklabels(pcf.columns, rotation=90)
ax.set_yticks(range(pcf.shape[0])); ax.set_yticklabels(tf_labels(pcf.index))
for i in range(pcf.shape[0]):
    for j in range(pcf.shape[1]):
        v = pcf.values[i, j]
        if v != 0: ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color='black')
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03); cb.set_label('TF motif accessibility (coef)')
cb.outline.set_linewidth(0.5)
ax.set_title('Perturbation vs NTC (pseudotime peaks)\nFDR-significant TF motif accessibility', fontsize=8)
savepanel(fig, 'FigS5F_PertTFActivity_heatmap')
